### Python Memory Management

- Memory management in python involves a combination of automatic garbage collection, reference counting, and various internal optimizations to efficiently manage memory allocation and deallocation. Understanding these mechanisms can help developers write more efficient and robust applications.

    1. Key Concepts in Python Memory Management
    2. Memory Allocation and Deallocation
    3. Reference Counting
    4. Garbage Collection
    5. The gc Module
    6. Memory Management best practices

### Reference Counting

- Reference counting is the primary method python uses to manage memory. Each object in python maintains a count of references pointing to it. When the reference count drops to zero, the memory occupied by the object is deallocated.

In [1]:
import sys

a=[]
# 2 (one reference from 'a' and one from getrefcount())
print(sys.getrefcount(a))

2


In [2]:
b=a
print(sys.getrefcount(b))

3


In [3]:
del b
print(sys.getrefcount(a))

2


### Garbage Collection

- Python includes a cyclic garbage collector to handle reference cycles. Reference cycles occur when objects reference each other, preventing their reference counts from reaching zero.

In [4]:
import gc

# enable garbage collection
gc.enable()

In [5]:
gc.disable()

In [6]:
gc.collect()

6

In [7]:
# Get garbage collection stats
print(gc.get_stats())

[{'collections': 185, 'collected': 1718, 'uncollectable': 0}, {'collections': 16, 'collected': 333, 'uncollectable': 0}, {'collections': 2, 'collected': 6, 'uncollectable': 0}]


In [8]:
# get unreachable objects
print(gc.garbage)

[]


### Memory Management Best Practices

1. Use Local Variables: Local variables have a shorter lifespan and are freed sooner than global variables.
2. Avoid Circular References: Circular references can lead to memory leaks if not properly managed.
3. Use Generators. Generators produce items one at a time and only keep one item in memory at a time, making them memory efficient. 
4. Explicitly Delete Objects: Use the del statement to delete variables and objects explicitly.
5. Profile Memory Usage: Use memory profiling tools like tracemalloc and memory_profiler to identify memory leaks and optimize memory usage.

In [9]:
# Handled Circular Reference
import gc

class MyObject:
    def __init__(self, name):
        self.name = name
        print(f"Object {self.name} created")
    
    def __del__(self):
        print(f"Object {self.name} deleted")

# Create circular reference
obj1 = MyObject('obj1')
obj2 = MyObject('obj2')

obj1.ref = obj2
obj2.ref = obj1

del obj1
del obj2

# Manually trigger the garbage collection
gc.collect()

Object obj1 created
Object obj2 created
Object obj1 deleted
Object obj2 deleted


2

In [10]:
# Handled Circular Reference - Sample 2(beautifying the output for easy output understanding)
import gc

class MyObject:
    def __init__(self, name):
        self.name = name
        print(f"Object {self.name} created")
    
    def __del__(self):
        print(f"Object {self.name} deleted")

# Create circular reference
obj1 = MyObject('obj1')
obj2 = MyObject('obj2')

obj1.ref = obj2
obj2.ref = obj1

# Remove references
del obj1
del obj2

# Check if objects are collectable before triggering GC
print(f"\nGarbage objects before collection: {len(gc.garbage)}")
print(f"Objects in generation 0: {gc.get_count()[0]}")

# Manually trigger the garbage collection
print("\n--- Running garbage collection ---")
collected = gc.collect()

print(f"Garbage collected: {collected} objects")
print(f"Garbage objects after collection: {len(gc.garbage)}")

# Enable debugging to see what's happening
print("\n--- With GC debugging ---")
gc.set_debug(gc.DEBUG_SAVEALL)  # Save garbage to gc.garbage
gc.collect()
print(f"Objects in gc.garbage: {len(gc.garbage)}")
if gc.garbage:
    print("These objects couldn't be collected due to __del__ methods")

Object obj1 created
Object obj2 created

Garbage objects before collection: 0
Objects in generation 0: 620

--- Running garbage collection ---
Object obj1 deleted
Object obj2 deleted
Garbage collected: 9 objects
Garbage objects after collection: 0

--- With GC debugging ---
Objects in gc.garbage: 0


In [11]:
# Generator for memory efficiency
# Generators allow you to produce items one at a time, using memory efficiently by only keeping one item in memory at a time.

def generate_numbers(n):
    for i in range(n):
        yield i

# Using the generator
for num in generate_numbers(100000):
    print(num)
    if num>10:
        break

0
1
2
3
4
5
6
7
8
9
10
11


In [12]:
# Profiling Memory usage with tracemalloc
import tracemalloc

def create_list():
    return [i for i in range(10000)]

def main():
    tracemalloc.start()

    my_list=create_list()

    snapshot = tracemalloc.take_snapshot()
    top_stats = snapshot.statistics('lineno')

    print("[Top 10]")
    for stat in top_stats[:10]:
       print(stat)


In [13]:
if __name__=='__main__':
    main()

[Top 10]
C:\Users\mani2\AppData\Local\Temp\ipykernel_34340\3664580024.py:5: size=388 KiB, count=9744, average=41 B


In [14]:
# Example of beautifying the output for easy reference
# import tracemalloc
import sys

def create_list():
    """Create a large list to measure memory"""
    return [i * 2 for i in range(100000)]  # 100,000 elements

# Clear any previous traces
tracemalloc.stop()

# Start tracing
tracemalloc.start()

# Memory before
snapshot1 = tracemalloc.take_snapshot()

# Create and store the list
my_big_list = create_list()

# Memory after
snapshot2 = tracemalloc.take_snapshot()

# Compare snapshots
top_stats = snapshot2.compare_to(snapshot1, 'lineno')

print("Memory allocated by create_list():")
for stat in top_stats[:5]:  # Show top 5 differences
    print(f"Size: {stat.size_diff / 1024:.2f} KB")
    print(f"Count: {stat.count_diff} allocations")
    print(f"File: {stat.traceback[-1].filename}")
    print(f"Line: {stat.traceback[-1].lineno}")
    print("-" * 50)

# Get current memory usage
current, peak = tracemalloc.get_traced_memory()
print(f"\nCurrent memory usage: {current / 1024:.2f} KB")
print(f"Peak memory usage: {peak / 1024:.2f} KB")

# Clean up
del my_big_list
tracemalloc.stop()

Memory allocated by create_list():
Size: 3903.12 KB
Count: 99872 allocations
File: C:\Users\mani2\AppData\Local\Temp\ipykernel_34340\1483625592.py
Line: 7
--------------------------------------------------
Size: 0.30 KB
Count: 2 allocations
File: c:\Users\mani2\AppData\Local\Programs\Python\Python311\Lib\tracemalloc.py
Line: 560
--------------------------------------------------
Size: 0.30 KB
Count: 2 allocations
File: c:\Users\mani2\AppData\Local\Programs\Python\Python311\Lib\tracemalloc.py
Line: 423
--------------------------------------------------
Size: 0.12 KB
Count: 2 allocations
File: c:\Users\mani2\AppData\Local\Programs\Python\Python311\Lib\threading.py
Line: 320
--------------------------------------------------
Size: 0.12 KB
Count: 1 allocations
File: C:\Users\mani2\AppData\Roaming\Python\Python311\site-packages\traitlets\traitlets.py
Line: 1514
--------------------------------------------------

Current memory usage: 3913.54 KB
Peak memory usage: 3928.13 KB
